# Zero-Train Optimization of a Healthcare Chatbot
Medical Chatbot:

Background: Medical advice chatbot (LLM) classifier model trained on based on Llama3.2 on online medical forums.


Dataset: https://huggingface.co/datasets/lextale/FirstAidInstructionsDataset/viewer/default/icliniqDataset


Scenario: Optimize the model without additional training.


Use Case: The model is already trained and we want to optimize it without additional training.

## Setup

### Install dependencies and utility functions: This cell should be run once.

In [1]:
%%capture

%pip install matplotlib numpy pandas requests tqdm ipywidgets
%pip install --ignore-installed 'git+https://github.com/Authentrics-ai/authentrics-client.git@v2.1.0'

from pathlib import Path

import os
import authentrics_client as authrx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import random
from datetime import datetime
import time
from tqdm.notebook import tqdm
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from threading import Thread
import contextlib

# Utility functions for loading indicators
@contextlib.contextmanager
def loading_spinner(message="Processing..."):
    """Context manager for showing a loading spinner"""
    spinner_widget = widgets.HTML(value=f"""
    <div style="display: flex; align-items: center; gap: 10px;">
        <div style="width: 20px; height: 20px; border: 2px solid #f3f3f3; border-top: 2px solid #3498db; border-radius: 50%; animation: spin 1s linear infinite;"></div>
        <span style="font-size: 14px; color: #555;">{message}</span>
    </div>
    <style>
    @keyframes spin {{
        0% {{ transform: rotate(0deg); }}
        100% {{ transform: rotate(360deg); }}
    }}
    </style>
    """)
    display(spinner_widget)
    try:
        yield
    finally:
        spinner_widget.close()

def show_progress_bar(total, description="Progress"):
    """Create and return a progress bar"""
    return tqdm(total=total, desc=description, bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]')

def wait_for_analytics(project_id, min_wait_time=30, max_wait_time=300, check_interval=10):
    """
    Wait for ALL analytics to be complete before continuing.
    This prevents NaN values from appearing in the summary table.
    """
    import time

    print("🔄 Waiting for analytics to process...")
    countdown_widget = widgets.HTML()
    display(countdown_widget)

    # Minimum wait with countdown
    for remaining in range(min_wait_time, 0, -1):
        countdown_widget.value = f"""
        <div style="display: flex; align-items: center; gap: 10px; margin: 10px 0;">
            <div style="width: 20px; height: 20px; border: 2px solid #f3f3f3; border-top: 2px solid #3498db; border-radius: 50%; animation: spin 1s linear infinite;"></div>
            <span style="font-size: 14px; color: #555;">Minimum wait period: {remaining} seconds remaining...</span>
        </div>
        <style>
        @keyframes spin {{
            0% {{ transform: rotate(0deg); }}
            100% {{ transform: rotate(360deg); }}
        }}
        </style>
        """
        time.sleep(1)

    # Poll until ALL analytics are complete
    start_polling = time.time()
    attempt = 1

    while (time.time() - start_polling) < (max_wait_time - min_wait_time):
        try:
            project = client.project.get_project_by_id(project_id)
            files = project["fileList"][1:]  # Skip base model

            # Check that ALL files have both weight and bias contributions
            files_missing_analytics = []
            for f in files:
                if (f.get("totalWeightContribution") is None or
                    f.get("totalBiasContribution") is None):
                    files_missing_analytics.append(f["fileName"])

            # Only continue if ALL analytics are complete
            if not files_missing_analytics:
                countdown_widget.value = """
                <div style="color: green; font-weight: bold; margin: 10px 0;">
                    ✅ All analytics are complete!
                </div>
                """
                return True

            # Show progress with specific missing files
            total_files = len(files)
            completed_files = total_files - len(files_missing_analytics)
            progress_pct = (completed_files / total_files * 100) if total_files > 0 else 0

            missing_display = ', '.join(files_missing_analytics[:3])
            if len(files_missing_analytics) > 3:
                missing_display += f" and {len(files_missing_analytics) - 3} more"

            countdown_widget.value = f"""
            <div style="display: flex; align-items: center; gap: 10px; margin: 10px 0;">
                <div style="width: 20px; height: 20px; border: 2px solid #f3f3f3; border-top: 2px solid #orange; border-radius: 50%; animation: spin 1s linear infinite;"></div>
                <span style="font-size: 14px; color: #555;">
                    Attempt {attempt}: {completed_files}/{total_files} files complete ({progress_pct:.1f}%)
                    <br/>⏳ Waiting for: {missing_display}
                </span>
            </div>
            <style>
            @keyframes spin {{
                0% {{ transform: rotate(0deg); }}
                100% {{ transform: rotate(360deg); }}
            }}
            </style>
            """
            attempt += 1
            time.sleep(check_interval)

        except Exception as e:
            print(f"⚠️ Error checking analytics status: {e}")
            time.sleep(check_interval)

    # Timeout reached - show final status
    try:
        project = client.project.get_project_by_id(project_id)
        files = project["fileList"][1:]
        files_missing_analytics = [f["fileName"] for f in files
                                 if (f.get("totalWeightContribution") is None or
                                     f.get("totalBiasContribution") is None)]

        if files_missing_analytics:
            countdown_widget.value = f"""
            <div style="color: red; font-weight: bold; margin: 10px 0;">
                ⚠️ Timeout: Analytics incomplete for {len(files_missing_analytics)} files.
                <br/>NaN values will appear for: {', '.join(files_missing_analytics[:5])}
                <br/>Consider increasing wait time or contacting support.
            </div>
            """
            return False
        else:
            countdown_widget.value = """
            <div style="color: green; font-weight: bold; margin: 10px 0;">
                ✅ All analytics completed just in time!
            </div>
            """
            return True
    except Exception as e:
        countdown_widget.value = f"""
        <div style="color: red; font-weight: bold; margin: 10px 0;">
            ❌ Error checking final status: {e}
        </div>
        """
        return False

### Contact Authentrics for the URL and user credentials info@authentrics.ai

In [ ]:

DEMO_SERVER_URL = input("Enter your Authentrics URL: ")
PROJECT_NAME = "Healthcare Chatbot ZTO"
pd.options.display.precision = 3
pd.options.display.chop_threshold = None

### Establish a session with authentrics
- Rerun this cell if your session expires

In [3]:
client = authrx.AuthentricsClient(DEMO_SERVER_URL)
client.auth.login()
print("✅ Session established successfully!")

✅ Session established successfully!


### Generate a project in Authentrics to track checkpoints, and perform asynchronous analytics
1. Creates a project for checkpoint management.
2. **Option A**: Point to existing checkpoint storage (no data movement required).
3. **Option B**: Upload checkpoints directly through our client (automated storage and versioning).

In [4]:
projects = client.project.get_projects()
if projects is not None:
    project = next((p for p in projects if p["name"].startswith(PROJECT_NAME)), None)
    if project:
        client.project.delete_project(project["id"], hard_delete=True)

PROJECT_NAME = PROJECT_NAME + " " + str(int(datetime.now().timestamp()))
project = client.project.create_project(
    PROJECT_NAME,
    "A smaller LLM specializing in medical advice",
    authrx.FileType.HF_TEXT,
)
project_id = project["id"]

### Restore all checkpoint files, stimulus files, and expected output file in bucket to a stable version

In [ ]:
with loading_spinner("Please wait while we restore the requested files..."):
    client.get('/transfer/MedChat')

### Point authentrics file registry to the checkpoints using the api

In [ ]:
print("Adding checkpoints to project...")
with show_progress_bar(5, "Adding checkpoints") as pbar:
    for i in range(20, 25):
        client.checkpoint.add_external_checkpoint(
            project_id,
            f"demo-models/Medical-Chatbot/iteration_{i}.tar",
            authrx.FileType.HF_TEXT,
            file_name=f"iteration_from_dataset_{i}.tar",
            tag=f"v{i}",
        )
        pbar.update(1)

project = client.project.get_project_by_name(PROJECT_NAME)
project_id = project["id"]
print(f"✅ Project created with id: {project['id']}")
print(f"Project name: {project['name']}")
for file in project["fileList"]:
    print(f"File name: {file['fileName']}")

Adding checkpoints to project...


Adding checkpoints:   0%|          | 0/20 [00:00<?]

✅ Project created with id: 69163aa2839b09178f2c1d63
Project name: Healthcare Chatbot 1763064481
File name: iteration_from_dataset_0.tar
File name: iteration_from_dataset_1.tar
File name: iteration_from_dataset_2.tar
File name: iteration_from_dataset_3.tar
File name: iteration_from_dataset_4.tar
File name: iteration_from_dataset_5.tar
File name: iteration_from_dataset_6.tar
File name: iteration_from_dataset_7.tar
File name: iteration_from_dataset_8.tar
File name: iteration_from_dataset_9.tar
File name: iteration_from_dataset_10.tar
File name: iteration_from_dataset_11.tar
File name: iteration_from_dataset_12.tar
File name: iteration_from_dataset_13.tar
File name: iteration_from_dataset_14.tar
File name: iteration_from_dataset_15.tar
File name: iteration_from_dataset_16.tar
File name: iteration_from_dataset_17.tar
File name: iteration_from_dataset_18.tar
File name: iteration_from_dataset_19.tar


Wait for Analytics to be ready before proceeding.

In [7]:
print("\n" + "="*60)
print("📊 ANALYTICS PROCESSING")
print("="*60)
print("Authentrics is now computing analytics in the background.")
print("This includes weight contributions and bias scores for each checkpoint.")

analytics_ready = wait_for_analytics(project_id, min_wait_time=30, max_wait_time=300)
if not analytics_ready:
    print("\n❌ Analytics are not complete.")
    print("Please wait a moment longer and then continue. Please contact support if the issue persists.")


📊 ANALYTICS PROCESSING
Authentrics is now computing analytics in the background.
This includes weight contributions and bias scores for each checkpoint.
🔄 Waiting for analytics to process...


## Zero-Train Optimization (ZTO) Results

Authentrics.ai provides a zero-train optimization (ZTO) service that allows you to optimize a model's performance without additional training.

This will increase or decrease the effects of each training iteration on the model to obtain the best possible performance. In this case, the performance is measured by the semantic similarity between the model's response and the expected output.

The scaling factor limit is the maximum fraction of change that any one training iteration can undergo.

In [ ]:
stimulus_paths = [f"demo-models/Medical-Chatbot/stimuli/prompt_{i:02d}.json" for i in range(10)]
expected_output_path = "demo-models/Medical-Chatbot/stimuli/expected_output.txt"

with loading_spinner("Running ZTO..."):
    result = client.dynamic.zero_train_optimizer(
        project_id=project["id"],
        scaling_factor_limit=0.1,
        stimulus_paths=stimulus_paths,
        batch_size=4,
        expected_output_path=expected_output_path,
        inference_config={"max_new_tokens": 100},
    )

print(result)